# Road Damage Detection (Pothole / Crack / Manhole)

โน้ตบุ๊กนี้ศึกษาการตรวจจับหลุมบ่อ รอยแตก และฝาท่อด้วย YOLO11 ภายใต้ภาพต้นฉบับและภาพแสงน้อยสังเคราะห์ ครอบคลุมการเตรียมข้อมูล การทดลอง 4 รูปแบบ การประเมินผล และต้นแบบตรวจจับจากกล้องพร้อมแจ้งเตือน

## 1. การตรวจสอบและทำความสะอาดข้อมูล

- ตรวจสอบรูปแบบไฟล์ ความละเอียด โหมดสี และความสามารถในการอ่านภาพ
- ตรวจภาพซ้ำด้วย MD5 และตัดออก 5 ภาพ เหลือ 2,004 ภาพ
- ตรวจสอบไฟล์ label รูปแบบ annotation และ class ID
- ปรับพิกัดที่เกินขอบภาพให้อยู่ในช่วง [0,1]
- ตัด annotation ที่ผิดรูปแบบ คลาสไม่ถูกต้อง หรือมีขนาดเล็กจนไม่มีพื้นที่ใช้งาน
- ทำความสะอาด label แบบ HBB และ polygon 4 จุดแยกกัน โดยไม่แก้ไฟล์ต้นฉบับ
- ใช้ชื่อคลาส `pothole`, `crack` และ `manhole` ตลอดการทดลอง

## 2. การแบ่ง Train / Validation / Test แบบกลุ่ม

จัดกลุ่มภาพจากส่วนต้นของชื่อไฟล์และตัวเลขท้ายชื่อไฟล์หารแบบปัดลงด้วย `BLOCK = 30` จากนั้นสุ่มกลุ่มด้วย `seed = 42` และแบ่งให้ใกล้เคียงสัดส่วน 80/10/10

| ชุดข้อมูล | จำนวนภาพต้นฉบับ |
|---|---:|
| Train | 1,602 |
| Validation | 201 |
| Test | 201 |

ข้อมูลแบ่งเป็น 1,751 กลุ่ม แต่ละกลุ่มอยู่ใน split เดียว และตรวจสอบว่าไม่มีภาพซ้ำข้าม split

ภาพต้นฉบับและภาพแสงน้อยที่สร้างจากภาพเดียวกันอยู่ใน split เดียวกันเสมอ

> การแบ่งแบบกลุ่มช่วยลดความเสี่ยงของข้อมูลรั่วไหล แต่ไม่ได้รับประกันว่าแต่ละ split มาจากคนละวิดีโอ

## 3. การสร้างภาพแสงน้อยสังเคราะห์

สร้างภาพแสงน้อย 1 ภาพต่อภาพต้นฉบับ โดยใช้:

| เทคนิค | ช่วงค่าที่สุ่ม |
|---|---|
| Gamma Correction | 1.4–2.2 |
| Brightness Scaling | 0.5–0.9 |
| ส่วนเบี่ยงเบนมาตรฐานของ Gaussian Noise | 0.01–0.025 |

ใช้ seed ที่คำนวณจากชื่อไฟล์ร่วมกับ `SEED` เพื่อให้สร้างภาพซ้ำได้เหมือนเดิม

การปรับแสงไม่เปลี่ยนตำแหน่งวัตถุ จึงใช้ annotation ที่ทำความสะอาดแล้วชุดเดียวกันกับภาพต้นฉบับ โดยไม่มีการใช้ CLAHE

E3 และ E4 ใช้ข้อมูลผสมทั้งสองชุด:
- Train: 3,204 ภาพ = Original 1,602 + Low-Light 1,602
- Validation: 402 ภาพ = Original 201 + Low-Light 201

## 4. การทดลองและการตั้งค่าฝึก

| การทดลอง | โมเดล | รูปแบบกรอบ | ข้อมูล Train และ Validation | Epochs สูงสุด | Patience |
|---|---|---|---|---:|---:|
| E1 | YOLO11n | HBB | Original | 50 | 20 |
| E2 | YOLO11s | HBB | Original | 150 | 50 |
| E3 | YOLO11s | HBB | Original + Low-Light | 150 | 50 |
| E4 | YOLO11s-OBB | OBB | Original + Low-Light | 150 | 50 |

**การตั้งค่าร่วม**
- ใช้ pretrained weights
- Image size: 640
- Random seed: 42
- เปิด deterministic training
- เลือกอุปกรณ์อัตโนมัติ: CUDA → Apple MPS → CPU
- Batch size: CUDA ใช้ 12, MPS ใช้ 8 และ CPU ใช้ 4
- DataLoader workers: Windows ใช้ 0 และระบบอื่นใช้ 4
- ตั้ง `QUICK_TEST = False` และ `RUN_EXPERIMENTS = None`

**การตั้งค่าเพิ่มเติมสำหรับ E2–E4**
- Cosine learning rate
- Rotation: ±5 องศา
- Scale: 0.6
- Translation: 0.1
- HSV value augmentation: 0.5
- Horizontal flipping: โอกาส 0.5
- ไม่ใช้ vertical flipping
- Mosaic: โอกาส 1.0 และปิดในช่วง 15 epochs สุดท้าย

ใช้ `best.pt` ที่เลือกจากผลบน Validation สำหรับประเมินผลสุดท้าย หากพบ `best.pt` ของการทดลองนั้นแล้ว โค้ดจะข้ามการฝึก

> E1 และ E2 ต่างกันทั้งขนาดโมเดลและการตั้งค่าฝึก จึงไม่ได้วัดผลของขนาดโมเดลเพียงอย่างเดียว

## 5. การประเมินผล

ทุกการทดลองใช้ภาพทดสอบชุดเดียวกัน:
- `test_orig`: ภาพต้นฉบับ 201 ภาพ
- `test_low`: ภาพแสงน้อยสังเคราะห์จากภาพเดียวกัน 201 ภาพ

ประเมินด้วย Precision, Recall, mAP@50, mAP@50–95 และ AP@50 รายคลาส

| การทดลอง | mAP@50–95 Original | mAP@50–95 Low-Light |
|---|---:|---:|
| E1 | 0.193 | 0.114 |
| E2 | 0.205 | 0.088 |
| E3 | 0.199 | 0.177 |
| E4 | 0.359 | 0.309 |

เมื่อเพิ่มภาพแสงน้อยในการฝึก E3 มี mAP@50–95 บน Low-Light สูงขึ้นจาก E2 ขณะที่ผลบน Original ลดลงเล็กน้อย ส่วน E4 มีค่า mAP@50–95 สูงที่สุดที่รายงานในชุดการทดลองนี้

> HBB และ OBB ใช้รูปแบบ annotation และการประเมินการซ้อนทับต่างกัน จึงไม่ควรเทียบคะแนนโดยตรงเพื่อสรุปว่าโมเดลใดเหนือกว่า

โค้ดเลือกการทดลองที่มี mAP@50–95 บน `test_orig` สูงที่สุดสำหรับสร้างกราฟและตัวอย่างภาพเพิ่มเติม การเลือกนี้ใช้ผล Test เพื่อวิเคราะห์ ส่วน checkpoint ของแต่ละการทดลองเลือกจาก Validation

## 6. กราฟและไฟล์ผลลัพธ์

บันทึกกราฟ PNG ใน `workspace/figures/` ได้แก่:
- ตัวอย่างภาพ การกระจายความสว่าง และความละเอียดภาพ
- จำนวนภาพและวัตถุในแต่ละ split
- ขนาดกรอบและตัวอย่าง annotation
- ภาพ Original เทียบกับ Synthetic Low-Light
- Training curves: Loss, Precision/Recall, mAP และ Learning Rate
- ผล Test และ AP@50 รายคลาส
- Confusion Matrix และ PR Curve
- การเปรียบเทียบ TTA และตัวอย่างภาพทำนาย

บันทึกตารางผลที่:
- `workspace/runs/results_summary.csv`
- `workspace/runs/tta_summary.csv`

ส่วน TTA เปรียบเทียบ `augment=False` กับ `augment=True` โดยไม่ได้รับประกันว่าผลจะดีขึ้น ส่วนการจูน hyperparameter เป็นโค้ดที่คอมเมนต์ไว้และยังไม่รัน

## 7. ต้นแบบตรวจจับจากกล้องและแจ้งเตือน

โหลด `road_damage_obb_best.pt` และรับภาพจากกล้องโทรศัพท์ผ่าน OpenCV

- อ่านภาพด้วย thread แยกและเก็บเฉพาะเฟรมล่าสุดเพื่อลดดีเลย์สะสม
- ตรวจจับด้วย `conf = 0.25`, `imgsz = 640` และ `device = "mps"`
- แสดงกรอบ OBB พร้อมข้อความภาษาไทย
- กำหนดระดับเตือนจากพื้นที่กรอบตรงที่ครอบ OBB เทียบกับพื้นที่ภาพ
- เลือกวัตถุที่มีความสำคัญสูงสุดสำหรับแจ้งเตือนในแต่ละเฟรม
- เล่นเสียงภาษาไทยที่สร้างไว้ล่วงหน้าผ่าน `pygame`
- เว้นการเตือนคลาสและระดับเดิมอย่างน้อย 3 วินาที
- ให้เสียงที่มีความสำคัญสูงกว่าขัดจังหวะเสียงเดิมได้
- กด `q` เพื่อปิดหน้าต่างตรวจจับ

> ระดับ “ใกล้มาก / ระวัง / ไกล” ประมาณจากขนาดกรอบในภาพ ไม่ใช่ระยะทางจริง โค้ดกล้องใช้ Apple MPS คำสั่ง `say` และฟอนต์ของ macOS

## 8. วิธีใช้งาน

1. เปิดโน้ตบุ๊กใน VS Code พร้อมส่วนขยาย Python และ Jupyter
2. เลือก Python environment และติดตั้งไลบรารีที่จำเป็น
3. ดาวน์โหลดข้อมูลด้วย `kagglehub` หรือกำหนด `RAW_OVERRIDE`
4. รันเซลล์เตรียมข้อมูล ฝึก และประเมินผลตามลำดับ
5. รันเซลล์กล้องแยก หลังเตรียมไฟล์โมเดล URL กล้อง และไลบรารีสำหรับระบบที่ใช้

> การทดลองใช้ภาพแสงน้อยสังเคราะห์ จึงต้องทดสอบเพิ่มเติมกับภาพถนนตอนกลางคืนจริงก่อนสรุปประสิทธิภาพในการใช้งานจริง

# **0. ติดตั้งและตั้งค่า**

In [ ]:
from pathlib import Path
requirements_path = Path("requirements.txt") if Path("requirements.txt").is_file() else Path("../requirements.txt")
%pip install -r {requirements_path}
# Install the appropriate PyTorch build for your device first; see README.


In [ ]:
import os, re, sys, random, shutil, hashlib, zlib, glob, platform
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch

# ---------------- Config (แก้ที่นี่ที่เดียว) ----------------
SEED = 42
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
WORK = REPO_ROOT / "workspace"          # โฟลเดอร์ทำงานทั้งหมด (อยู่ข้างไฟล์ notebook)
RAW_OVERRIDE = None                      # ถ้าโหลด dataset เองแล้ว ให้ใส่ path โฟลเดอร์ที่มี images/ กับ labels-YOLO/ เช่น Path(r"D:\datasets\road_damage\data")
KAGGLE_SLUG = "lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes"

DS      = WORK / "ds"
RUNS    = WORK / "runs"
FIG_DIR = WORK / "figures"
for d in (WORK, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {0: "pothole", 1: "crack", 2: "manhole"}
CLASS_COLORS = {"pothole": "#E45756", "crack": "#4C78A8", "manhole": "#54A24B"}

SPLIT_RATIO = (0.8, 0.1, 0.1)
BLOCK = 30        # เฟรมติดกัน BLOCK เฟรมถือเป็นกลุ่มเดียว (ไม่แยกข้าม train/val/test)
IMGSZ = 640       # ภาพต้นฉบับ 640x360 อยู่แล้ว; 960 ช้ากว่า ~2 เท่าและเกิน VRAM 6GB เมื่อ batch สูง
QUICK_TEST = False   # True = เทรนแค่ไม่กี่ epoch บนข้อมูลส่วนน้อย เพื่อเช็คว่าโค้ดรันครบทุกขั้น

# ---------------- เลือกอุปกรณ์อัตโนมัติ ----------------
if torch.cuda.is_available():
    DEVICE, BATCH = 0, 12                # RTX 4050 6GB @640: 12 ใช้ได้; ถ้าเจอ CUDA out of memory ให้ลดเป็น 8
    print("Device: CUDA ->", torch.cuda.get_device_name(0))
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE, BATCH = "mps", 8
    print("Device: Apple MPS")
else:
    DEVICE, BATCH = "cpu", 4
    print("⚠️ ไม่พบ GPU: จะรันบน CPU ซึ่งช้ามาก แนะนำตั้ง QUICK_TEST = True เพื่อลองโค้ด หรือลด IMGSZ/epochs")

WORKERS = 0 if os.name == "nt" else 4    # Windows + notebook: workers>0 มักค้าง

random.seed(SEED)
np.random.seed(SEED)

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

def save_fig(name):
    # เซฟกราฟเป็น PNG แล้วแสดงใน notebook
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved -> {FIG_DIR / (name + '.png')}")

# **0.1 ดึง Dataset**

เรียก `kagglehub` ให้ดาวน์โหลดอัตโนมัติ (ครั้งแรกอาจต้องล็อกอิน Kaggle: สร้าง API token ที่ kaggle.com → Settings → *Create New Token* แล้วเก็บไว้ที่ `~/.kaggle/kaggle.json` หรือสั่ง `kagglehub.login()`)

ถ้าโหลดไม่ได้: ดาวน์โหลด zip จากหน้า dataset เอง → แตกไฟล์ → ตั้ง `RAW_OVERRIDE` ให้ชี้โฟลเดอร์ที่มี `images/` และ `labels-YOLO/`

In [ ]:
def find_raw_root():
    if RAW_OVERRIDE:
        return Path(RAW_OVERRIDE)
    # เคยวางไว้ใน workspace/road_damage แล้วหรือไม่
    local = WORK / "road_damage"
    if local.exists():
        hit = next(local.rglob("labels-YOLO"), None)
        if hit:
            return hit.parent
    import kagglehub
    try:
        p = Path(kagglehub.dataset_download(KAGGLE_SLUG))
    except Exception as e:
        raise RuntimeError(
            "ดาวน์โหลดจาก Kaggle ไม่สำเร็จ: " + str(e) +
            "\nแก้โดย (1) ตั้งค่า Kaggle API token  หรือ (2) โหลด zip เองแล้วแตกไว้ที่ workspace/road_damage หรือตั้ง RAW_OVERRIDE"
        )
    hit = next(p.rglob("labels-YOLO"), None)
    if hit is None:
        raise FileNotFoundError(f"ไม่พบโฟลเดอร์ labels-YOLO ใน {p}")
    return hit.parent


RAW      = find_raw_root()
IMG_DIR  = RAW / "images"
LBL_HBB  = RAW / "labels-YOLO"   # กล่องตรง  : class xc yc w h
LBL_POLY = RAW / "labels"        # polygon 4 จุด : class x1 y1 ... x4 y4 (ใช้กับ OBB)
print("RAW:", RAW)
print("images dir exists:", IMG_DIR.exists(), "| labels-YOLO:", LBL_HBB.exists(), "| labels:", LBL_POLY.exists())

# **1. ตรวจสอบรูปภาพ**

In [ ]:
image_files = sorted(f for f in os.listdir(IMG_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png")))

formats, resolutions, modes, corrupted = Counter(), Counter(), Counter(), []
for f in image_files:
    try:
        with Image.open(IMG_DIR / f) as im:
            formats[im.format] += 1
            resolutions[im.size] += 1
            modes[im.mode] += 1
    except Exception:
        corrupted.append(f)

print("Total images :", len(image_files))
print("Formats      :", dict(formats))
print("Resolutions  :", dict(resolutions))
print("Modes        :", dict(modes))
print("Corrupted    :", len(corrupted))

In [ ]:
# ภาพซ้ำ (md5) — เก็บไฟล์แรก (ตามลำดับชื่อ) ลบที่เหลือออกจากการใช้งาน
hashes = defaultdict(list)
for f in image_files:
    if f in corrupted:
        continue
    hashes[hashlib.md5((IMG_DIR / f).read_bytes()).hexdigest()].append(f)

duplicates_to_remove = [f for fs in hashes.values() if len(fs) > 1 for f in fs[1:]]
clean_images = [f for f in image_files if f not in set(duplicates_to_remove) and f not in corrupted]

print("Duplicate images excluded:", len(duplicates_to_remove), duplicates_to_remove)
print("Clean images             :", len(clean_images))

In [ ]:
brightness = [np.array(Image.open(IMG_DIR / f).convert("L")).mean() for f in clean_images]
print(f"Brightness min/mean/max: {min(brightness):.1f} / {np.mean(brightness):.1f} / {max(brightness):.1f}")

plt.figure(figsize=(12, 8))
for i, f in enumerate(random.sample(clean_images, min(9, len(clean_images)))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(Image.open(IMG_DIR / f)); plt.title(f[:24]); plt.axis("off")
plt.tight_layout()
save_fig("01_sample_images")

In [ ]:
# กราฟ: การกระจายความสว่าง + ความละเอียดของภาพ
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(brightness, bins=30, color="#4C78A8", edgecolor="white")
ax[0].axvline(np.mean(brightness), color="red", ls="--", label=f"mean = {np.mean(brightness):.1f}")
ax[0].set(title="Image brightness distribution (original)", xlabel="Mean gray level (0-255)", ylabel="# images")
ax[0].legend()

top = resolutions.most_common(8)
labels = [f"{w}x{h}" for (w, h), _ in top]
counts = [c for _, c in top]
bars = ax[1].bar(labels, counts, color="#F58518")
ax[1].bar_label(bars)
ax[1].set(title="Image resolution", ylabel="# images")
ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
save_fig("02_brightness_resolution")

# **2. ตรวจและทำความสะอาด Annotation**

ฟังก์ชันด้านล่างจะ
- ตัดบรรทัดที่ format ผิด / class ไม่อยู่ใน {0,1,2}
- **clip** พิกัดที่ล้นขอบภาพให้อยู่ใน [0,1] (แทนการลบทิ้งทั้งกล่อง)
- ตัดกล่อง/polygon ที่เล็กจนไม่มีพื้นที่ (degenerate)

ทำกับทั้ง label กล่องตรง (HBB) และ polygon (OBB) โดย **ไม่แก้ไฟล์ต้นฉบับ** — ผลลัพธ์ถูกเขียนตอนสร้าง dataset

In [ ]:
def clean_hbb_lines(text, stats):
    out = []
    for ln in text.splitlines():
        p = ln.split()
        if not p:
            continue
        if len(p) != 5:
            stats["bad_format"] += 1; continue
        try:
            c = int(float(p[0])); x, y, w, h = map(float, p[1:])
        except ValueError:
            stats["bad_format"] += 1; continue
        if c not in CLASS_NAMES:
            stats["bad_class"] += 1; continue
        box = (x - w / 2, y - h / 2, x + w / 2, y + h / 2)
        cb = tuple(min(max(v, 0.0), 1.0) for v in box)
        if any(abs(a - b) > 1e-6 for a, b in zip(box, cb)):
            stats["clipped"] += 1
        nw, nh = cb[2] - cb[0], cb[3] - cb[1]
        if nw < 1e-3 or nh < 1e-3:
            stats["degenerate"] += 1; continue
        out.append(f"{c} {(cb[0] + cb[2]) / 2:.6f} {(cb[1] + cb[3]) / 2:.6f} {nw:.6f} {nh:.6f}")
    return out


def poly_area(pts):
    xs, ys = pts[0::2], pts[1::2]
    return 0.5 * abs(sum(xs[i] * ys[(i + 1) % 4] - xs[(i + 1) % 4] * ys[i] for i in range(4)))


def clean_poly_lines(text, stats):
    out = []
    for ln in text.splitlines():
        p = ln.split()
        if not p:
            continue
        if len(p) != 9:
            stats["bad_format"] += 1; continue
        try:
            c = int(float(p[0])); pts = list(map(float, p[1:]))
        except ValueError:
            stats["bad_format"] += 1; continue
        if c not in CLASS_NAMES:
            stats["bad_class"] += 1; continue
        cl = [min(max(v, 0.0), 1.0) for v in pts]
        if any(abs(a - b) > 1e-6 for a, b in zip(pts, cl)):
            stats["clipped"] += 1
        if poly_area(cl) < 1e-6:
            stats["degenerate"] += 1; continue
        out.append(f"{c} " + " ".join(f"{v:.6f}" for v in cl))
    return out


TASKS = {"hbb": (LBL_HBB, clean_hbb_lines), "obb": (LBL_POLY, clean_poly_lines)}

# ตรวจทั้งชุด (dry run) เพื่อรายงานจำนวนปัญหา
raw_class_counts = {}
for task, (ldir, fn) in TASKS.items():
    stats, missing, cls_count, empty = Counter(), 0, Counter(), 0
    for f in clean_images:
        lp = ldir / (Path(f).stem + ".txt")
        if not lp.exists():
            missing += 1; continue
        lines = fn(lp.read_text(), stats)
        empty += (len(lines) == 0)
        for ln in lines:
            cls_count[int(ln.split()[0])] += 1
    raw_class_counts[task] = cls_count
    print(f"[{task}] missing label files: {missing} | empty after clean: {empty} | issues: {dict(stats)}")
    print(f"[{task}] objects per class:", {CLASS_NAMES[k]: v for k, v in sorted(cls_count.items())})

# **3. แบ่ง Train / Val / Test (Grouped split)**

ชื่อไฟล์ `vlcsnap-xxxxx` คือเฟรมที่ตัดจากวิดีโอ เฟรมที่อยู่ใกล้กันแทบเหมือนกัน
ถ้าสุ่มแบ่งทีละภาพ เฟรมคล้ายกันจะไปอยู่ทั้ง train และ test ทำให้ผลดูดีเกินจริง
จึงรวมเฟรมที่เลขติดกันเป็น block แล้วสุ่มแบ่งทีละ block

In [ ]:
def group_key(stem):
    m = re.match(r"^(.*?)(\d+)$", stem)
    return (m.group(1), int(m.group(2)) // BLOCK) if m else (stem, 0)


groups = defaultdict(list)
for f in clean_images:
    groups[group_key(Path(f).stem)].append(f)

keys = sorted(groups, key=str)
random.Random(SEED).shuffle(keys)

n = len(clean_images)
names = ["train", "val", "test"]
target = dict(zip(names, [r * n for r in SPLIT_RATIO]))
split = {s: [] for s in names}
# greedy: ส่งแต่ละกลุ่มไปยัง split ที่ยัง "ขาด" สัดส่วนมากที่สุด -> ได้สัดส่วนใกล้เป้าหมาย ไม่มี split ว่าง
for k in keys:
    s = max(names, key=lambda x: (target[x] - len(split[x])) / target[x])
    split[s] += groups[k]
assert all(len(v) > 0 for v in split.values()), "มี split ว่าง: ลด BLOCK แล้วรันใหม่"

print("groups:", len(groups))
for s, fs in split.items():
    print(f"{s:5s}: {len(fs)} images")

# ตรวจว่าไม่มี overlap
sets = {s: set(fs) for s, fs in split.items()}
assert not (sets["train"] & sets["val"]) and not (sets["train"] & sets["test"]) and not (sets["val"] & sets["test"])
print("No overlap between splits ✅")

In [ ]:
# กราฟ: จำนวนภาพต่อ split
sizes = [len(split[s]) for s in names]
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(names, sizes, color=["#4C78A8", "#F58518", "#54A24B"])
ax.bar_label(bars, labels=[f"{v}\n({v / sum(sizes) * 100:.1f}%)" for v in sizes])
ax.set(title="Images per split (grouped split)", ylabel="# images")
ax.set_ylim(0, max(sizes) * 1.15)
plt.tight_layout()
save_fig("03_split_sizes")

# **4. สร้าง Low-light (สังเคราะห์) และ Dataset จริง**

- Low-light สุ่มระดับ gamma / brightness / noise **ต่อภาพ** (ใช้ seed จากชื่อไฟล์ → ได้ผลซ้ำเดิมทุกครั้ง)
- ภาพ original กับ low-light ของภาพเดียวกันอยู่ split เดียวกันเสมอ
- โครงสร้าง: `images/{train,val,test}_{orig,low}` (label ใช้ชุดเดียวกัน)
- สร้างทั้ง HBB (`workspace/ds/hbb`) และ OBB (`workspace/ds/obb`)

In [ ]:
def make_lowlight(arr_uint8, rng):
    img = arr_uint8.astype(np.float32) / 255.0
    gamma = rng.uniform(1.4, 2.2)
    bright = rng.uniform(0.5, 0.9)
    noise = rng.uniform(0.01, 0.025)
    out = np.power(img, gamma) * bright
    out = out + rng.normal(0, noise, out.shape)
    return (np.clip(out, 0, 1) * 255).astype(np.uint8)


def write_label(path, lines):
    path.write_text("\n".join(lines) + ("\n" if lines else ""))


shutil.rmtree(DS, ignore_errors=True)
for task in TASKS:
    for s in split:
        for v in ("orig", "low"):
            (DS / task / "images" / f"{s}_{v}").mkdir(parents=True, exist_ok=True)
            (DS / task / "labels" / f"{s}_{v}").mkdir(parents=True, exist_ok=True)

build_stats = {t: Counter() for t in TASKS}
for s, files in split.items():
    for f in files:
        stem = Path(f).stem
        arr = np.array(Image.open(IMG_DIR / f).convert("RGB"))
        rng = np.random.default_rng((zlib.crc32(stem.encode()) + SEED) % (2**32))
        low = make_lowlight(arr, rng)

        for task, (ldir, fn) in TASKS.items():
            lp = ldir / f"{stem}.txt"
            if not lp.exists():
                build_stats[task]["missing_label"] += 1
            lines = fn(lp.read_text() if lp.exists() else "", build_stats[task])

            root = DS / task
            # original
            shutil.copy2(IMG_DIR / f, root / "images" / f"{s}_orig" / f)
            write_label(root / "labels" / f"{s}_orig" / f"{stem}.txt", lines)
            # low-light
            Image.fromarray(low).save(root / "images" / f"{s}_low" / f"{stem}_low.jpg", quality=95)
            write_label(root / "labels" / f"{s}_low" / f"{stem}_low.txt", lines)

for t, st in build_stats.items():
    print(t, dict(st))
for task in TASKS:
    for s in split:
        for v in ("orig", "low"):
            ni = len(os.listdir(DS / task / "images" / f"{s}_{v}"))
            nl = len(os.listdir(DS / task / "labels" / f"{s}_{v}"))
            assert ni == nl, (task, s, v, ni, nl)
print("Dataset built, image/label counts match ✅")

In [ ]:
import yaml


def make_yamls(task, train_variants):
    '''train_variants: 'orig' = เทรนเฉพาะภาพปกติ, 'both' = ปกติ + low-light
    คืน dict {'orig': yaml_for_test_orig, 'low': yaml_for_test_low}'''
    tv = ["orig"] if train_variants == "orig" else ["orig", "low"]
    out = {}
    for test_v in ("orig", "low"):
        d = {
            "path": (DS / task).as_posix(),
            "train": [f"images/train_{v}" for v in tv],
            "val":   [f"images/val_{v}" for v in tv],
            "test":  f"images/test_{test_v}",
            "names": CLASS_NAMES,
        }
        p = DS / f"{task}_{train_variants}_test_{test_v}.yaml"
        p.write_text(yaml.dump(d, sort_keys=False))
        out[test_v] = p
    return out


print(open(make_yamls("hbb", "both")["orig"]).read())

## 4.1 กราฟวิเคราะห์ Dataset ที่สร้างเสร็จ

In [ ]:
# กราฟ: จำนวน object ต่อคลาสในแต่ละ split + ขนาดกล่อง (อ่านจาก label HBB ของภาพ original)
rows = []
for s in split:
    for p in (DS / "hbb" / "labels" / f"{s}_orig").glob("*.txt"):
        for ln in p.read_text().splitlines():
            c, x, y, w, h = ln.split()
            rows.append({"split": s, "cls": CLASS_NAMES[int(c)], "w": float(w), "h": float(h)})
df_boxes = pd.DataFrame(rows)
df_boxes["area"] = df_boxes["w"] * df_boxes["h"]
print(df_boxes.groupby("cls")[["w", "h", "area"]].describe().T.round(4).iloc[:, :3] if len(df_boxes) else "no boxes")

if len(df_boxes):
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))

    # (a) object per class per split
    ct = pd.crosstab(df_boxes["cls"], df_boxes["split"])[names]
    ct.plot(kind="bar", ax=ax[0], color=["#4C78A8", "#F58518", "#54A24B"], edgecolor="white", rot=0)
    for cont in ax[0].containers:
        ax[0].bar_label(cont, fontsize=8)
    ax[0].set(title="Objects per class in each split", xlabel="", ylabel="# objects")

    # (b) w vs h
    for cname, g in df_boxes.groupby("cls"):
        ax[1].scatter(g["w"], g["h"], s=8, alpha=0.35, label=cname, color=CLASS_COLORS[cname])
    ax[1].set(title="Normalized box size (width vs height)", xlabel="width", ylabel="height", xlim=(0, 1), ylim=(0, 1))
    ax[1].legend(markerscale=2)

    # (c) area distribution
    bins = np.logspace(np.log10(max(df_boxes["area"].min(), 1e-6)), 0, 30)
    for cname, g in df_boxes.groupby("cls"):
        ax[2].hist(g["area"], bins=bins, alpha=0.55, label=cname, color=CLASS_COLORS[cname])
    ax[2].set_xscale("log")
    ax[2].set(title="Box area distribution (log scale)", xlabel="area (fraction of image)", ylabel="# objects")
    ax[2].legend()

    plt.tight_layout()
    save_fig("04_dataset_class_and_box_stats")

In [ ]:
# กราฟ: เทียบภาพ original vs low-light + histogram ความสว่าง
sample = random.sample(split["train"], min(4, len(split["train"])))
fig, axes = plt.subplots(2, len(sample), figsize=(4 * len(sample), 6), squeeze=False)
for j, f in enumerate(sample):
    stem = Path(f).stem
    axes[0, j].imshow(Image.open(DS / "hbb/images/train_orig" / f)); axes[0, j].set_title("original", fontsize=9)
    axes[1, j].imshow(Image.open(DS / "hbb/images/train_low" / f"{stem}_low.jpg")); axes[1, j].set_title("low-light (synthetic)", fontsize=9)
    for i in (0, 1):
        axes[i, j].axis("off")
plt.tight_layout()
save_fig("05_original_vs_lowlight")

sub = random.sample(split["train"], min(150, len(split["train"])))
b_orig = [np.array(Image.open(DS / "hbb/images/train_orig" / f).convert("L")).mean() for f in sub]
b_low  = [np.array(Image.open(DS / "hbb/images/train_low" / f"{Path(f).stem}_low.jpg").convert("L")).mean() for f in sub]
plt.figure(figsize=(7, 4))
plt.hist(b_orig, bins=25, alpha=0.6, label=f"original (mean {np.mean(b_orig):.0f})", color="#F58518")
plt.hist(b_low, bins=25, alpha=0.6, label=f"low-light (mean {np.mean(b_low):.0f})", color="#4C78A8")
plt.title("Brightness: original vs synthetic low-light"); plt.xlabel("Mean gray level"); plt.ylabel("# images"); plt.legend()
plt.tight_layout()
save_fig("06_brightness_orig_vs_low")

# **5. (แนะนำ) ดูคุณภาพ Label ด้วยตา**

วาดทั้งกล่องตรง (เขียว) และ polygon (แดง) บนภาพที่มี crack — ดูว่า label สม่ำเสมอไหม
ถ้า crack ถูกติดเป็นท่อน ๆ หรือกล่องใหญ่เกินจริง การแก้ label จะช่วยได้มากกว่าการปรับ hyperparameter

In [ ]:
import cv2


def draw_gt(img_path, hbb_path, poly_path):
    im = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = im.shape[:2]
    for ln in hbb_path.read_text().splitlines():
        c, x, y, w, h = ln.split(); x, y, w, h = map(float, (x, y, w, h))
        cv2.rectangle(im, (int((x - w / 2) * W), int((y - h / 2) * H)),
                          (int((x + w / 2) * W), int((y + h / 2) * H)), (0, 255, 0), 1)
    for ln in poly_path.read_text().splitlines():
        p = ln.split(); pts = np.array(list(map(float, p[1:]))).reshape(4, 2) * [W, H]
        cv2.polylines(im, [pts.astype(np.int32)], True, (255, 0, 0), 1)
        cv2.putText(im, CLASS_NAMES[int(p[0])], tuple(pts[0].astype(int)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
    return im


crack_imgs = [f for f in split["train"]
              if any(l.startswith("1 ") for l in (DS / "hbb/labels/train_orig" / f"{Path(f).stem}.txt").read_text().splitlines())]
sel = random.sample(crack_imgs, min(6, len(crack_imgs)))

if sel:
    plt.figure(figsize=(16, 8))
    for i, f in enumerate(sel):
        st = Path(f).stem
        plt.subplot(2, 3, i + 1)
        plt.imshow(draw_gt(DS / "hbb/images/train_orig" / f, DS / "hbb/labels/train_orig" / f"{st}.txt",
                           DS / "obb/labels/train_orig" / f"{st}.txt"))
        plt.title(f"{f}  (green=HBB, red=polygon)", fontsize=9); plt.axis("off")
    plt.tight_layout()
    save_fig("07_label_quality_check")
else:
    print("ไม่พบภาพที่มี crack ใน train")

# **6. การทดลอง (ใช้ split เดียว วัด test เดียวกัน)**

| ชื่อ | Task | โมเดล | ข้อมูลเทรน | Config |
|---|---|---|---|---|
| E1 | HBB | YOLO11n | original | baseline เดิม (640, 50 ep) |
| E2 | HBB | YOLO11s | original | improved |
| E3 | HBB | YOLO11s | original + low-light | improved |
| E4 | OBB | YOLO11s-obb | original + low-light | improved |

ทุกโมเดลถูกวัดบน **test_orig** และ **test_low** (ภาพชุดเดียวกัน ต่างแค่ความสว่าง)

> ใน notebook นี้เปิดไว้เฉพาะ **E4** (เหมือนไฟล์ต้นฉบับ) — ถ้าอยากเทียบหลายโมเดล ให้เอา `#` หน้า E1–E3 ออกในเซลล์ถัดไป
> - ถ้ามี GPU ระดับ T4 การเทรน E2–E4 ใช้เวลาหลายชั่วโมง (imgsz 960) → ลด `IMGSZ` เป็น 800/640 หรือกำหนด `RUN_EXPERIMENTS`
> - ถ้าเจอ CUDA out of memory ให้ลด `BATCH` ในเซลล์ config
> - ถ้ายังไม่แน่ใจว่าโค้ดรันได้ครบ ตั้ง `QUICK_TEST = True` ก่อน (เทรนแค่ 2 epoch บนข้อมูล 10%)

In [ ]:
from ultralytics import YOLO

COMMON = dict(imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True, device=DEVICE, workers=WORKERS,
              project=str(RUNS), exist_ok=True, plots=True)

BASELINE = dict(epochs=50, patience=20, imgsz=640)
IMPROVED = dict(epochs=150, patience=50, cos_lr=True, close_mosaic=15,
                degrees=5.0, scale=0.6, translate=0.1, hsv_v=0.5,
                fliplr=0.5, flipud=0.0, mosaic=1.0)

EXPERIMENTS = [
    dict(id="E1", name="E1_hbb_n_baseline",            task="hbb", weights="yolo11n.pt",     train="orig", cfg=BASELINE),
    dict(id="E2", name="E2_hbb_s_improved",            task="hbb", weights="yolo11s.pt",     train="orig", cfg=IMPROVED),
    dict(id="E3", name="E3_hbb_s_improved_lowlight",   task="hbb", weights="yolo11s.pt",     train="both", cfg=IMPROVED),
    dict(id="E4", name="E4_obb_s_improved_lowlight",   task="obb", weights="yolo11s-obb.pt", train="both", cfg=IMPROVED),
]
RUN_EXPERIMENTS = None   # None = ทั้งหมด หรือเช่น ["E1", "E3"]

if QUICK_TEST:
    RUNS = WORK / "runs_quicktest"          # แยกโฟลเดอร์ ไม่ปนกับผลเทรนจริง
    COMMON["project"] = str(RUNS)


def exp_cfg(exp):
    # รวม config ของการทดลอง (ใช้ทั้งตอนเทรนและตอนประเมิน เพื่อให้ imgsz ตรงกัน)
    cfg = {**COMMON, **exp["cfg"]}
    if QUICK_TEST:
        cfg.update(epochs=2, patience=2, fraction=0.1, imgsz=640, close_mosaic=0)
    return cfg


for exp in EXPERIMENTS:
    if RUN_EXPERIMENTS and exp["id"] not in RUN_EXPERIMENTS:
        continue
    best = RUNS / exp["name"] / "weights" / "best.pt"
    if best.exists():
        print(f"[skip] {exp['name']} มี best.pt แล้ว (ลบโฟลเดอร์ runs/{exp['name']} ถ้าต้องการเทรนใหม่)")
        continue
    print(f"\n===== Training {exp['name']} =====")
    data_yaml = make_yamls(exp["task"], exp["train"])["orig"]
    model = YOLO(exp["weights"])
    model.train(data=data_yaml.as_posix(), name=exp["name"], **exp_cfg(exp))

## 6.1 กราฟ Training curves

In [ ]:
def _col(df, prefix):
    # หาคอลัมน์ที่ขึ้นต้นด้วย prefix (ชื่อคอลัมน์ต่างกันเล็กน้อยตามเวอร์ชัน ultralytics)
    return next((c for c in df.columns if c.startswith(prefix)), None)


def load_results_csv(exp_name):
    p = RUNS / exp_name / "results.csv"
    if not p.exists():
        return None
    df = pd.read_csv(p)
    df.columns = [c.strip() for c in df.columns]
    return df


def plot_training_curves(exp_name):
    df = load_results_csv(exp_name)
    if df is None:
        print(f"[skip] ไม่พบ results.csv ของ {exp_name}")
        return
    ep = df["epoch"] if "epoch" in df else np.arange(1, len(df) + 1)
    c_map50, c_map = _col(df, "metrics/mAP50("), _col(df, "metrics/mAP50-95(")
    fitness = 0.1 * df[c_map50] + 0.9 * df[c_map]
    best_i = int(fitness.idxmax())

    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    for ax, key, title in zip(axes.flat[:3], ["box_loss", "cls_loss", "dfl_loss"], ["Box loss", "Class loss", "DFL loss"]):
        tr, va = _col(df, f"train/{key}"), _col(df, f"val/{key}")
        if tr: ax.plot(ep, df[tr], label="train", color="#4C78A8")
        if va: ax.plot(ep, df[va], label="val", color="#E45756")
        ax.set(title=title, xlabel="epoch", ylabel="loss"); ax.legend()

    ax = axes[1, 0]
    ax.plot(ep, df[_col(df, "metrics/precision")], label="Precision", color="#4C78A8")
    ax.plot(ep, df[_col(df, "metrics/recall")], label="Recall", color="#F58518")
    ax.set(title="Precision / Recall (val)", xlabel="epoch", ylim=(0, 1.02)); ax.legend()

    ax = axes[1, 1]
    ax.plot(ep, df[c_map50], label="mAP@50", color="#54A24B")
    ax.plot(ep, df[c_map], label="mAP@50-95", color="#B279A2")
    ax.axvline(ep.iloc[best_i], color="gray", ls="--", label=f"best epoch = {int(ep.iloc[best_i])}")
    ax.scatter([ep.iloc[best_i]], [df[c_map].iloc[best_i]], color="red", zorder=5)
    ax.set(title="mAP (val)", xlabel="epoch", ylim=(0, 1.02)); ax.legend()

    ax = axes[1, 2]
    lr = _col(df, "lr/pg0")
    if lr:
        ax.plot(ep, df[lr], color="#9D755D")
    ax.set(title="Learning rate", xlabel="epoch", ylabel="lr")

    fig.suptitle(f"Training curves - {exp_name}", fontsize=14, y=1.01)
    plt.tight_layout()
    save_fig(f"08_training_curves_{exp_name}")
    print(f"best epoch {int(ep.iloc[best_i])}: mAP50={df[c_map50].iloc[best_i]:.3f}  mAP50-95={df[c_map].iloc[best_i]:.3f}")


for exp in EXPERIMENTS:
    plot_training_curves(exp["name"])

# เทียบ mAP50-95 (val) ของทุกการทดลองในกราฟเดียว (ถ้ามีมากกว่า 1 การทดลอง)
curves = {e["name"]: load_results_csv(e["name"]) for e in EXPERIMENTS}
curves = {k: v for k, v in curves.items() if v is not None}
if len(curves) >= 2:
    plt.figure(figsize=(9, 5))
    for k, df in curves.items():
        plt.plot(df["epoch"], df[_col(df, "metrics/mAP50-95(")], label=k)
    plt.title("Validation mAP@50-95 across experiments"); plt.xlabel("epoch"); plt.ylabel("mAP@50-95"); plt.legend()
    plt.tight_layout()
    save_fig("09_training_compare_experiments")

# **7. ประเมินผลและเปรียบเทียบ**

In [ ]:
def evaluate(weights, yaml_path, imgsz, augment=False, plots=False, name=None):
    m = YOLO(str(weights))
    extra = dict(project=str(RUNS / "eval"), name=name, exist_ok=True) if name else {}
    r = m.val(data=str(yaml_path), split="test", imgsz=imgsz, batch=BATCH, device=DEVICE, workers=WORKERS,
              augment=augment, plots=plots, verbose=False, **extra)
    row = {"P": r.box.mp, "R": r.box.mr, "mAP50": r.box.map50, "mAP50-95": r.box.map}
    for i, c in enumerate(r.box.ap_class_index):
        row[f"AP50_{m.names[int(c)]}"] = r.box.ap50[i]
    return (row, Path(r.save_dir)) if plots else row


rows = []
for exp in EXPERIMENTS:
    best = RUNS / exp["name"] / "weights" / "best.pt"
    if not best.exists():
        continue
    imgsz = exp_cfg(exp)["imgsz"]
    ys = make_yamls(exp["task"], exp["train"])
    for test_v in ("orig", "low"):
        row = evaluate(best, ys[test_v], imgsz)
        rows.append({"exp": exp["name"], "test": test_v, **row})

results = pd.DataFrame(rows).round(3)
results.to_csv(RUNS / "results_summary.csv", index=False)
results

In [ ]:
# ตารางเปรียบเทียบแบบอ่านง่าย: P / R / mAP บน test original vs low-light
view = results.pivot(index="exp", columns="test", values=["P", "R", "mAP50", "mAP50-95"]).round(3)
view

In [ ]:
# กราฟ: เปรียบเทียบ P / R / mAP ของแต่ละการทดลอง บน test original vs low-light
def grouped_bar(ax, df, value_col, title):
    exps = list(dict.fromkeys(df["exp"]))
    x = np.arange(len(exps)); w = 0.38
    for k, (tv, color) in enumerate([("orig", "#F58518"), ("low", "#4C78A8")]):
        vals = [df[(df["exp"] == e) & (df["test"] == tv)][value_col].values[0] for e in exps]
        bars = ax.bar(x + (k - 0.5) * w, vals, w, label=f"test_{tv}", color=color)
        ax.bar_label(bars, fmt="%.3f", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels([e.split("_")[0] for e in exps])
    ax.set(title=title, ylim=(0, 1.05)); ax.legend(fontsize=8)


if len(results):
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
    for ax, col in zip(axes, ["P", "R", "mAP50", "mAP50-95"]):
        grouped_bar(ax, results, col, col)
    fig.suptitle("Test-set performance: original vs low-light", fontsize=13, y=1.03)
    plt.tight_layout()
    save_fig("10_test_metrics_orig_vs_low")

    # AP50 ต่อคลาส
    ap_cols = [c for c in results.columns if c.startswith("AP50_")]
    long = results.melt(id_vars=["exp", "test"], value_vars=ap_cols, var_name="cls", value_name="AP50")
    long["cls"] = long["cls"].str.replace("AP50_", "", regex=False)
    long["label"] = long["exp"].str.split("_").str[0] + " / " + long["test"]
    pv = long.pivot(index="cls", columns="label", values="AP50")
    ax = pv.plot(kind="bar", figsize=(10, 5), rot=0, edgecolor="white", colormap="tab20")
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%.2f", fontsize=7)
    ax.set(title="AP@50 per class (test set)", xlabel="", ylabel="AP@50", ylim=(0, 1.05))
    ax.legend(title="experiment / test", fontsize=8)
    plt.tight_layout()
    save_fig("11_ap50_per_class")

    # ความทนต่อแสงน้อย: mAP50-95 ตกลงเท่าไรเมื่อเปลี่ยน orig -> low
    drop = results.pivot(index="exp", columns="test", values="mAP50-95")
    drop["drop (orig-low)"] = drop["orig"] - drop["low"]
    print("\nmAP50-95 ที่ลดลงเมื่อทดสอบบนภาพ low-light:")
    print(drop.round(3))
else:
    print("ยังไม่มีผลประเมิน (ยังไม่มี best.pt)")

## 7.1 Confusion matrix และ PR curve ของโมเดลที่ดีที่สุด

In [ ]:
# Exploratory display choice based on recorded test scores; not validation-only model selection.
# เลือกโมเดลที่ดีที่สุดอัตโนมัติ = mAP50-95 สูงสุดบน test_orig
if len(results):
    BEST_EXP = results[results["test"] == "orig"].sort_values("mAP50-95", ascending=False).iloc[0]["exp"]
else:
    BEST_EXP = EXPERIMENTS[0]["name"]
# BEST_EXP = "E4_obb_s_improved_lowlight"   # ปลด comment แล้วระบุเองได้
print("BEST_EXP =", BEST_EXP)

exp = next(e for e in EXPERIMENTS if e["name"] == BEST_EXP)
imgsz = exp_cfg(exp)["imgsz"]
ys = make_yamls(exp["task"], exp["train"])
best = RUNS / exp["name"] / "weights" / "best.pt"

for test_v in ("orig", "low"):
    _, sd = evaluate(best, ys[test_v], imgsz, plots=True, name=f"{BEST_EXP}_test_{test_v}")
    imgs = [("confusion matrix", sd / "confusion_matrix_normalized.png")]
    imgs += [("PR curve", p) for p in sorted(sd.glob("*PR_curve.png"))[:1]]
    imgs = [(t, p) for t, p in imgs if p.exists()]
    if not imgs:
        print("ไม่พบไฟล์กราฟใน", sd); continue
    fig, axes = plt.subplots(1, len(imgs), figsize=(8 * len(imgs), 6), squeeze=False)
    for ax, (t, p) in zip(axes[0], imgs):
        ax.imshow(Image.open(p)); ax.axis("off"); ax.set_title(f"{t} - {BEST_EXP} on test_{test_v}")
    plt.tight_layout()
    save_fig(f"12_confusion_pr_test_{test_v}")

# **8. ขั้นตอนเสริม**

**8.1 TTA ตอนทดสอบ** (ช้าลงแต่มักได้เพิ่มเล็กน้อย)

In [ ]:
tta_rows = []
for test_v in ("orig", "low"):
    base = evaluate(best, ys[test_v], imgsz, augment=False)
    tta  = evaluate(best, ys[test_v], imgsz, augment=True)
    print(test_v, "TTA:", {k: round(float(v), 3) for k, v in tta.items()})
    tta_rows.append({"test": test_v, "mode": "no TTA", **base})
    tta_rows.append({"test": test_v, "mode": "TTA", **tta})
tta_df = pd.DataFrame(tta_rows).round(3)
tta_df.to_csv(RUNS / "tta_summary.csv", index=False)

# กราฟ: TTA vs no TTA
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, col in zip(axes, ["mAP50", "mAP50-95"]):
    x = np.arange(2); w = 0.38
    for k, (mode, color) in enumerate([("no TTA", "#9D9D9D"), ("TTA", "#54A24B")]):
        vals = [tta_df[(tta_df["test"] == t) & (tta_df["mode"] == mode)][col].values[0] for t in ("orig", "low")]
        bars = ax.bar(x + (k - 0.5) * w, vals, w, label=mode, color=color)
        ax.bar_label(bars, fmt="%.3f", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(["test_orig", "test_low"])
    ax.set(title=f"{col}: TTA vs no TTA ({BEST_EXP.split('_')[0]})", ylim=(0, 1.05)); ax.legend()
plt.tight_layout()
save_fig("13_tta_comparison")

**8.2 ดูภาพที่โมเดลทำนายบน test (หา failure case)**

In [ ]:
m = YOLO(str(best))
all_low = sorted(glob.glob(str(DS / exp["task"] / "images" / "test_low" / "*.jpg")))
all_orig = sorted(glob.glob(str(DS / exp["task"] / "images" / "test_orig" / "*.*")))
rs = random.Random(SEED)
paths = rs.sample(all_low, min(6, len(all_low)))
preds = m.predict(paths, imgsz=imgsz, conf=0.25, device=DEVICE, verbose=False)

plt.figure(figsize=(16, 8))
for i, r in enumerate(preds):
    plt.subplot(2, 3, i + 1)
    plt.imshow(r.plot()[..., ::-1]); plt.axis("off"); plt.title(Path(r.path).name, fontsize=9)
plt.suptitle("Predictions on test_low (conf >= 0.25)")
plt.tight_layout()
save_fig("14_predictions_test_low")

# เทียบภาพ original / low-light ของภาพเดียวกัน
stems = [Path(p).stem.replace("_low", "") for p in paths[:3]]
orig_map = {Path(p).stem: p for p in all_orig}
pairs = [(orig_map[s], str(DS / exp["task"] / "images" / "test_low" / f"{s}_low.jpg")) for s in stems if s in orig_map]
if pairs:
    flat = [p for pr in pairs for p in pr]
    pr_res = m.predict(flat, imgsz=imgsz, conf=0.25, device=DEVICE, verbose=False)
    fig, axes = plt.subplots(len(pairs), 2, figsize=(12, 4.2 * len(pairs)), squeeze=False)
    for i in range(len(pairs)):
        for j, tag in enumerate(["original", "low-light"]):
            axes[i, j].imshow(pr_res[2 * i + j].plot()[..., ::-1]); axes[i, j].axis("off"); axes[i, j].set_title(tag)
    plt.tight_layout()
    save_fig("15_predictions_orig_vs_low")

**8.3 (ไม่บังคับ) จูน hyperparameter อัตโนมัติ** — ใช้เวลานาน ปลด comment เมื่อพร้อม

In [ ]:
# model = YOLO("yolo11s.pt")
# model.tune(data=make_yamls("hbb", "both")["orig"].as_posix(), epochs=30, iterations=40,
#            imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=WORKERS, plots=False, save=False, val=False,
#            project=str(RUNS), name="tune")

# **สรุปวิธีอ่านผล**

- เทียบ **E1 → E2** = ผลของโมเดลใหญ่ขึ้น + เทรนนานขึ้น + imgsz/augmentation ใหม่
- เทียบ **E2 → E3** = ผลของการเพิ่มภาพ low-light ตอนเทรน (ดูคอลัมน์ test `low` ว่าดีขึ้นไหม)
- **E4 (OBB)** ใช้ metric คนละแบบ — เทียบเชิงแนวโน้ม ไม่ใช่ตัวเลขต่อตัวเลข
- ถ้า pothole/crack ยังต่ำ ให้กลับไปดูข้อ 5 (คุณภาพ label) และ failure cases ในข้อ 8.2
- กราฟทั้งหมดอยู่ที่ `workspace/figures/` และตารางผลอยู่ที่ `workspace/runs/results_summary.csv`

## Optional camera demonstration (separate script)

The original macOS Thai-voice camera prototype is preserved in `../examples/macos_camera_demo.py`. It is excluded from Run All. See the README for platform requirements and environment variables. Its size-based alert levels are **not measured distances** or a validated road-safety system.


Run the optional script explicitly from the repository root after training. Camera access is not needed for the research experiments.